#transform customer data

##remove null customer_id

In [0]:
%sql
select * from gizmobox.bronze.v_customer
where customer_id is not null

##remove exact duplicate records

In [0]:
%sql
select * from gizmobox.bronze.v_customer
where customer_id is not null
order by customer_id

In [0]:
%sql
select distinct * from gizmobox.bronze.v_customer
where customer_id is not null
order by customer_id

In [0]:
%sql
select 
    customer_id,
    max(created_timestamp), 
    max(customer_name),
    date_of_birth,
    max(email),
    max(member_since),
    max(telephone)
from gizmobox.bronze.v_customer
where customer_id is not null
group by customer_id, date_of_birth
order by customer_id

##cast the correct data_types
We can write using CTE

In [0]:
%sql
with cte_max as
(
    select customer_id,
    max(created_timestamp) as max_timestamp
    from gizmobox.bronze.v_customer
    group by customer_id
)
select distinct cast(c. max_timestamp as timestamp) as created_timestamp,
        c.customer_id,
        v.customer_name,
        cast(v.date_of_birth as date) as date_of_birth,
        v.email,
        v.member_since,
        v.telephone
from gizmobox.bronze.v_customer v
inner join cte_max c
on v.customer_id = c.customer_id
and v.created_timestamp = c.max_timestamp
order by c.customer_id

##write data to delta table

In [0]:
%sql
create table gizmobox.silver.customer
as
with cte_max as
(
    select customer_id,
    max(created_timestamp) as max_timestamp
    from gizmobox.bronze.v_customer
    group by customer_id
)
select distinct cast(c. max_timestamp as timestamp) as created_timestamp,
        c.customer_id,
        v.customer_name,
        cast(v.date_of_birth as date) as date_of_birth,
        v.email,
        v.member_since,
        v.telephone
from gizmobox.bronze.v_customer v
inner join cte_max c
on v.customer_id = c.customer_id
and v.created_timestamp = c.max_timestamp
order by c.customer_id

In [0]:
%sql
describe extended gizmobox.silver.customer 